# Trabalho Prático — versão Google Colab

Base: SF Bay Area Housing. Recorte: vendas registradas em 2005.


### Célula 01

Prepara o ambiente, baixa a base do Google Drive e importa as bibliotecas.


In [ ]:
from pathlib import Path
import sys
import subprocess

try:
    import gdown
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
    import gdown

DATA_FILE = Path('/content/sfhousing.txt')
DRIVE_FILE_ID = '10e8W6zxIqwgO2fCDjCsgTR8sv7PtXVL5'

if not DATA_FILE.exists():
    gdown.download(id=DRIVE_FILE_ID, output=str(DATA_FILE), quiet=False)

if not DATA_FILE.exists() or DATA_FILE.stat().st_size == 0:
    raise FileNotFoundError('Não foi possível baixar sfhousing.txt do Google Drive.')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix

plt.style.use('classic')
pd.set_option('display.float_format', '{:.2f}'.format)
print(f'Base disponível em: {DATA_FILE} ({DATA_FILE.stat().st_size:,} bytes)')


SSLError: HTTPSConnectionPool(host='drive.google.com', port=443): Max retries exceeded with url: /uc?id=10e8W6zxIqwgO2fCDjCsgTR8sv7PtXVL5 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: Basic Constraints of CA cert not marked critical (_ssl.c:1032)')))

### Célula 02

Carrega a base e seleciona as vendas registradas em 2005.


In [ ]:
df_orig = pd.read_csv(DATA_FILE, sep=',')
df_orig['date'] = pd.to_datetime(df_orig['date'], errors='coerce')

df = df_orig[df_orig['date'].dt.year == 2005].copy().reset_index(drop=True)

print(f'Dataset completo : {df_orig.shape[0]:,} registros')
print(f'Amostra de 2005  : {df.shape[0]:,} registros ({df.shape[0]/df_orig.shape[0]*100:.1f}% do total)')
df.head(5)


### Célula 03

Exibe a estrutura, os tipos e a quantidade de registros.


In [ ]:
df.info()


### Célula 04

Classifica a natureza e a unidade das variáveis.


In [ ]:
tipo_map = {
    'county'  : ('Categórico', 'Nominal', 'Condado'),
    'city'    : ('Categórico', 'Nominal', 'Cidade'),
    'zip'     : ('Categórico', 'Nominal', 'Identificador postal'),
    'street'  : ('Texto', 'Nominal', 'Endereço'),
    'price'   : ('Numérico (float)', 'Quantitativo contínuo', 'Dólares'),
    'br'      : ('Numérico (float)', 'Quantitativo discreto', 'Contagem'),
    'lsqft'   : ('Numérico (float)', 'Quantitativo contínuo', 'Pés quadrados'),
    'bsqft'   : ('Numérico (float)', 'Quantitativo contínuo', 'Pés quadrados'),
    'year'    : ('Numérico (float)', 'Quantitativo discreto', 'Ano'),
    'date'    : ('Data', 'Temporal', 'Data da venda'),
    'datesold': ('Data', 'Temporal', 'Segunda data de venda'),
}
pd.DataFrame(tipo_map, index=['Tipo no arquivo', 'Natureza', 'Observação']).T


### Célula 05

Calcula a quantidade e o percentual de valores ausentes.


In [ ]:
missing = pd.DataFrame({
    'ausentes': df.isna().sum(),
    '% ausentes': (df.isna().mean() * 100).round(2)
})
missing[missing['ausentes'] > 0]


### Célula 06

Detecta duplicidades e aplica critérios operacionais de plausibilidade.


In [ ]:
num_cols = ['price', 'br', 'lsqft', 'bsqft', 'year']
key_cols = ['city', 'zip', 'street', 'date']

duplicadas = df.duplicated(key_cols, keep=False)
print('Duplicidades segundo a chave informada no enunciado:',
      duplicadas.sum(), 'linhas envolvidas')
display(df.loc[duplicadas].sort_values(key_cols))

print()
print('Valores extremos antes da limpeza:')
print(df[num_cols].agg(['min', 'max']).to_string())

regras = {
    'ano fora de 1800–2005': df['year'].notna() & ~df['year'].between(1800, 2005),
    'terreno acima de 500.000 pés²': df['lsqft'].notna() & df['lsqft'].gt(500_000),
    'área construída acima de 20.000 pés²': df['bsqft'].notna() & df['bsqft'].gt(20_000),
    'mais de 20 quartos': df['br'].notna() & df['br'].gt(20),
}
for nome, mascara in regras.items():
    print(f'{nome}: {mascara.sum():,} registro(s)')

mascara_ruido = np.logical_or.reduce(list(regras.values()))
n_antes = len(df)
df = df.loc[~mascara_ruido].copy().reset_index(drop=True)

print()
print(f'Registros separados pelos critérios: {n_antes - len(df):,} ({(n_antes-len(df))/n_antes*100:.2f}%)')
print(f'Registros mantidos na análise      : {len(df):,}')


### Célula 07

Calcula medidas de posição, dispersão, assimetria e curtose.


In [ ]:
dt = df[num_cols].describe()
dt.loc['variance'] = df[num_cols].var()
dt.loc['cv'] = dt.loc['std'] / dt.loc['mean']
dt.loc['range'] = df[num_cols].max() - df[num_cols].min()
dt.loc['siqr'] = (dt.loc['75%'] - dt.loc['25%']) / 2
dt.loc['skewness'] = df[num_cols].skew()
dt.loc['kurtosis'] = df[num_cols].kurt()

ordem = ['count', 'mean', '50%', 'std', 'variance', 'cv', 'min', '25%',
         '75%', 'max', 'range', 'siqr', 'skewness', 'kurtosis']
dt.loc[ordem].T.round(2)


### Célula 08

Gera histogramas com curva normal teórica.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 5.6))
axes = axes.ravel()

for ax, col in zip(axes, num_cols):
    s = df[col].dropna()
    ax.hist(s, bins=40, density=True, color='steelblue',
            edgecolor='white', alpha=0.75)
    if s.std(ddof=0) > 0:
        x = np.linspace(s.min(), s.max(), 400)
        mu, sigma = s.mean(), s.std(ddof=0)
        normal = (1/(sigma*np.sqrt(2*np.pi))) * np.exp(-0.5*((x-mu)/sigma)**2)
        ax.plot(x, normal, color='firebrick', linewidth=1.4)
    ax.set_title(f'{col} | assimetria={df[col].skew():.2f}', fontsize=9)
    ax.tick_params(labelsize=7)
    ax.set_ylabel('Densidade', fontsize=8)

axes[-1].axis('off')
fig.suptitle('Distribuicoes das variaveis numericas', fontsize=12)
plt.tight_layout()
plt.show()


### Célula 09

Classifica a forma das distribuições a partir da assimetria calculada.


In [ ]:

assimetria = df[num_cols].skew()

def classificar_distribuicao(valor):
    intensidade = abs(valor)
    lado = 'Right-skewed' if valor >= 0 else 'Left-skewed'
    if intensidade < 1:
        return f'{lado} leve'
    if intensidade >= 5:
        return f'{lado} extremo'
    return lado

tipos_distribuicao = pd.DataFrame({
    'Forma': [classificar_distribuicao(assimetria[col]) for col in num_cols],
    'Evidência': [f'skew = {assimetria[col]:.2f}'.replace('.', ',') for col in num_cols]
}, index=num_cols)
tipos_distribuicao.index.name = 'Variável'

print('Tipos de distribuição identificados:')
display(
    tipos_distribuicao.style
    .set_properties(**{'text-align': 'left'})
    .set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'left'),
                                     ('background-color', '#eeeeee'),
                                     ('color', '#222222')]},
        {'selector': 'td', 'props': [('border', '1px solid #bdbdbd')]}
    ])
)


### Célula 10

Exibe a curtose e a indicação de inspeção de extremos na mesma ordem do arquivo de entrega.


In [ ]:
print('Curtose (valor positivo sugere caudas mais pesadas que a normal):')
for col in num_cols:
    k = df[col].kurt()
    indicacao = 'inspecionar extremos' if k > 0 else 'sem indicação por este critério'
    print(f'  {col:<8}: κ={k:>8.2f}  {indicacao}')


Curtose (valor positivo sugere caudas mais pesadas que a normal):


NameError: name 'num_cols' is not defined

### Célula 11

Gera boxplots das variáveis numéricas.


In [ ]:
rotulos = {
    'price': 'Preço de venda (US$)', 'br': 'Quartos',
    'lsqft': 'Área do terreno (pé²)', 'bsqft': 'Área construída (pé²)',
    'year': 'Ano de construção'
}

fig, axes = plt.subplots(3, 2, figsize=(10, 7.4))
axes = axes.ravel()

for ax, col in zip(axes, num_cols):
    valores = df[col].dropna()
    ax.boxplot(valores, showmeans=True, orientation='horizontal', widths=0.52,
               flierprops={'marker': '.', 'markersize': 2, 'alpha': 0.22},
               meanprops={'marker': 'D', 'markersize': 4,
                          'markerfacecolor': 'darkorange', 'markeredgecolor': 'black'})
    ax.set_title(rotulos[col], fontsize=10, pad=8)
    ax.set_yticks([])
    ax.tick_params(axis='x', labelsize=8)
    ax.grid(axis='x', alpha=0.25)
    if col == 'price':
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'US$ {x/1e6:.1f} mi'))
    elif col in ('lsqft', 'bsqft'):
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f} mil'))

axes[-1].axis('off')
fig.suptitle('Valores atípicos nas variáveis numéricas', fontsize=13, y=0.995)
fig.legend(handles=[plt.Line2D([], [], marker='D', linestyle='', markersize=5,
                              markerfacecolor='darkorange', markeredgecolor='black',
                              label='Média')],
           loc='lower right', bbox_to_anchor=(0.92, 0.08), frameon=False)
plt.tight_layout(rect=[0, 0.03, 1, 0.96], h_pad=1.5, w_pad=1.8)
plt.show()


### Célula 12

Compara a dispersão completa com a faixa de preços abaixo de US$ 4 milhões.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7.2), sharex=False)

axes[0].scatter(df['bsqft'], df['price'], alpha=0.12, s=7,
                color='steelblue', edgecolors='none')
axes[0].set_title('Amostra completa: efeito visual dos valores extremos', pad=9)
axes[0].set_xlabel('Área construída (pé²)')
axes[0].set_ylabel('Preço de venda (US$)')

limite_visual = 4_000_000
df_sem_cauda = df[df['price'] < limite_visual]
axes[1].scatter(df_sem_cauda['bsqft'], df_sem_cauda['price'], alpha=0.14, s=7,
                color='darkorange', edgecolors='none')
axes[1].set_title('Ampliação da faixa principal: preços abaixo de US$ 4 milhões', pad=9)
axes[1].set_xlabel('Área construída (pé²)')
axes[1].set_ylabel('Preço de venda (US$)')

for ax in axes:
    ax.grid(alpha=0.25)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'US$ {x/1e6:.1f} mi'))
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f} mil'))

fig.suptitle('Preço de venda × área construída', fontsize=13, y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.96], h_pad=2.0)
plt.show()

q1, q3 = df['price'].quantile([0.25, 0.75])
limite_iqr = q3 + 1.5 * (q3 - q1)
print(f'Limite superior pelo critério de 1,5×IQR: US$ {limite_iqr:,.0f}')
print(f'Registros acima desse limite: {(df["price"] > limite_iqr).sum():,}')
print(f'Registros com preço >= US$ 4 milhões: {(df["price"] >= limite_visual).sum():,}')


### Célula 13

Compara média e mediana nas distribuições.


In [ ]:
rotulos_distrib = {
    'price': 'Preço de venda', 'br': 'Número de quartos',
    'lsqft': 'Área do terreno', 'bsqft': 'Área construída',
    'year': 'Ano de construção'
}
fig, axes = plt.subplots(2, 3, figsize=(11, 7.2))
axes = axes.ravel()

for ax, col in zip(axes, num_cols):
    s = df[col].dropna()
    s.plot(kind='hist', bins=40, density=True, color='steelblue',
           edgecolor='white', alpha=0.76, ax=ax)
    media, mediana = s.mean(), s.median()
    ax.axvline(media, color='firebrick', lw=2, linestyle='--')
    ax.axvline(mediana, color='darkgreen', lw=2, linestyle='-')
    ax.set_title(rotulos_distrib[col], fontsize=10, pad=8)
    ax.set_ylabel('Densidade', fontsize=8)
    ax.grid(axis='y', alpha=0.2)
    ax.tick_params(labelsize=8)

    if col == 'price':
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'US$ {x/1e6:.1f} mi'))
        formato = lambda v: f'US$ {v/1e6:.2f} mi'
    elif col in ('lsqft', 'bsqft'):
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f} mil'))
        formato = lambda v: f'{v/1000:.1f} mil pé²'
    elif col == 'year':
        ax.ticklabel_format(axis='x', style='plain', useOffset=False)
        formato = lambda v: f'{v:.0f}'
    else:
        formato = lambda v: f'{v:.2f}'

    ax.text(0.98, 0.94, f'Média: {formato(media)}\nMediana: {formato(mediana)}',
            transform=ax.transAxes, ha='right', va='top', fontsize=8,
            bbox={'boxstyle': 'round,pad=0.3', 'facecolor': 'white',
                  'edgecolor': '#cccccc', 'alpha': 0.9})

axes[-1].axis('off')
legenda = [
    plt.Line2D([], [], color='firebrick', lw=2, linestyle='--', label='Média'),
    plt.Line2D([], [], color='darkgreen', lw=2, linestyle='-', label='Mediana')
]
axes[-1].legend(handles=legenda, loc='center', frameon=False, fontsize=10,
                title='Medidas de posição', title_fontsize=10)
fig.suptitle('Média e mediana por variável', fontsize=14, y=0.99)
plt.tight_layout(rect=[0, 0, 1, 0.95], h_pad=2.0, w_pad=1.7)
plt.show()


### Célula 14

Gera gráficos de dispersão em escalas original e logarítmica.


In [ ]:
sub = df[['price','bsqft','lsqft','br']].dropna()
sub = sub[(sub['price'] < 4_000_000) & (sub['bsqft'] < 10_000) & (sub['lsqft'] < 100_000)]

fig, axes = plt.subplots(3, 2, figsize=(10, 10.2))
graficos = [
    ('Relação linear: preço × área construída', sub['bsqft'], sub['price'],
     'Área construída (pé²)', 'Preço de venda (US$)', 'steelblue'),
    ('Relação log-log: preço × área construída', np.log10(sub['bsqft']), np.log10(sub['price']),
     'log10 da área construída', 'log10 do preço', 'darkorange'),
    ('Relação log-linear: preço × área construída', np.log10(sub['bsqft']), sub['price'],
     'log10 da área construída', 'Preço de venda (US$)', 'seagreen'),
    ('Relação log-log: área construída × terreno', np.log10(sub['lsqft']), np.log10(sub['bsqft']),
     'log10 da área do terreno', 'log10 da área construída', 'purple'),
    ('Relação linear: preço × número de quartos', sub['br'], sub['price'],
     'Número de quartos', 'Preço de venda (US$)', 'coral'),
]

for ax, (titulo, x, y, xlabel, ylabel, cor) in zip(axes.ravel(), graficos):
    ax.scatter(x, y, alpha=0.12, s=7, color=cor, edgecolors='none')
    ax.set_title(titulo, fontsize=10, pad=8)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.grid(alpha=0.25)
    if ylabel == 'Preço de venda (US$)':
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'US$ {v/1e6:.1f} mi'))

axes[-1, -1].axis('off')
axes[-1, -1].text(0.5, 0.55, 'Escalas comparadas', ha='center', va='center',
                  fontsize=12, fontweight='bold')
axes[-1, -1].text(0.5, 0.38, 'linear • log-linear • log-log', ha='center', va='center', fontsize=10)
fig.suptitle('Relacionamentos entre variáveis e efeito da escala', fontsize=14, y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.97], h_pad=2.0, w_pad=1.5)
plt.show()


### Célula 15

Gera a matriz de dispersão das variáveis numéricas.


In [ ]:
from matplotlib.ticker import ScalarFormatter

nomes_matriz = {
    'price': 'Preço (US$)', 'br': 'Quartos', 'lsqft': 'Terreno (pé²)',
    'bsqft': 'Construída (pé²)', 'year': 'Ano'
}
matriz_df = df[num_cols].rename(columns=nomes_matriz)
if len(matriz_df) > 20_000:
    matriz_df = matriz_df.sample(20_000, random_state=42)

mat_axes = scatter_matrix(
    matriz_df, figsize=(11, 10), range_padding=0.08, diagonal='hist',
    color='steelblue', alpha=0.10, s=7,
    hist_kwds={'bins': 35, 'color': 'steelblue', 'alpha': 0.82, 'edgecolor': 'white'}
)
for ax in np.asarray(mat_axes).ravel():
    ax.tick_params(axis='both', labelsize=6, pad=1)
    ax.xaxis.label.set_rotation(0)
    ax.yaxis.label.set_rotation(90)
    ax.xaxis.label.set_size(8)
    ax.yaxis.label.set_size(8)
    ax.grid(alpha=0.15)

    if ax.get_xlabel() == 'Ano':
        fmt_x = ScalarFormatter(useOffset=False)
        fmt_x.set_scientific(False)
        ax.xaxis.set_major_formatter(fmt_x)
    if ax.get_ylabel() == 'Ano':
        fmt_y = ScalarFormatter(useOffset=False)
        fmt_y.set_scientific(False)
        ax.yaxis.set_major_formatter(fmt_y)

plt.suptitle('Matriz de dispersão das variáveis numéricas', fontsize=14, y=1.01)
plt.tight_layout(pad=1.2)
plt.show()


### Célula 16

Resume a escala e a classificação exploratória observadas nos gráficos de dispersão.


In [ ]:

classificacao_relacionamentos = pd.DataFrame({
    'Par': ['price × bsqft', 'lsqft × bsqft', 'price × br', 'price × year'],
    'Escala em que o padrão ficou mais próximo de uma reta': [
        'log(price) × log(bsqft)',
        'log(lsqft) × log(bsqft)',
        'escala original',
        'escala original'
    ],
    'Classificação exploratória': [
        'Log-Log',
        'Log-Log',
        'Aproximadamente linear, com grande dispersão',
        'Sem relação evidente'
    ]
})

print('Leitura dos gráficos com base na Tabela 2 do resumo:')
display(
    classificacao_relacionamentos.style
    .hide(axis='index')
    .set_properties(**{'text-align': 'left'})
    .set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'left'),
                                     ('background-color', '#eeeeee'),
                                     ('color', '#222222')]},
        {'selector': 'td', 'props': [('border', '1px solid #d0d0d0')]}
    ])
)


### Célula 17

Calcula e exibe as correlações de Pearson e Spearman.


In [ ]:
corr_p = df[num_cols].corr('pearson')
corr_s = df[num_cols].corr('spearman')

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))

sns.heatmap(corr_p, cmap='RdBu_r', vmin=-1, vmax=1, annot=True, fmt='.2f',
            linewidths=0.5, ax=axes[0])
axes[0].set_title('Correlação de Pearson')

sns.heatmap(corr_s, cmap='RdBu_r', vmin=-1, vmax=1, annot=True, fmt='.2f',
            linewidths=0.5, ax=axes[1])
axes[1].set_title('Correlação de Spearman')

plt.suptitle('Heatmaps de correlação — variáveis numéricas', fontsize=13)
plt.tight_layout()
plt.show()


### Célula 18

Verifica pares de variáveis explicativas com correlação absoluta acima de 0,75.


In [ ]:
print('Pares de variáveis explicativas com |correlação| > 0,75:')
print()

ind_cols = ['br', 'lsqft', 'bsqft', 'year']
found = False
linhas_pares = []

for metodo, corr in [('Pearson', corr_p), ('Spearman', corr_s)]:
    print(f'--- {metodo} ---')
    for i in range(len(ind_cols)):
        for j in range(i + 1, len(ind_cols)):
            r = corr.loc[ind_cols[i], ind_cols[j]]
            classe = 'forte' if abs(r) > 0.75 else 'fraca ou moderada'
            par = f'{ind_cols[i]} × {ind_cols[j]}'
            print(f'  {ind_cols[i]:<8} x {ind_cols[j]:<8}: r={r:+.2f}  {classe}')
            linhas_pares.append({
                'Método': metodo,
                'Par': par,
                'Correlação': f'r = {r:+.2f}'.replace('.', ','),
                'Classificação': classe
            })
            found = found or abs(r) > 0.75
    print()

if not found:
    print('Nenhum par numérico ultrapassou o limite de 0,75 adotado no resumo.')

tabela_pares = pd.DataFrame(linhas_pares)
display(
    tabela_pares.style
    .hide(axis='index')
    .set_properties(**{'text-align': 'left'})
    .set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'left'),
                                     ('background-color', '#eeeeee'),
                                     ('color', '#222222')]},
        {'selector': 'td', 'props': [('border', '1px solid #d0d0d0')]}
    ])
)


### Célula 19

Compara distribuições originais e transformadas em log10.


In [ ]:
transf_cols = ['price', 'lsqft', 'bsqft']
rotulos_transf = {
    'price': 'Preço de venda', 'lsqft': 'Área do terreno',
    'bsqft': 'Área construída'
}

fig, axes = plt.subplots(len(transf_cols), 2, figsize=(10, 9.2))

for i, col in enumerate(transf_cols):
    s = df[col].dropna()
    s_log = np.log10(s[s > 0])

    axes[i, 0].hist(s, bins=50, density=True, color='steelblue',
                    edgecolor='white', alpha=0.82)
    axes[i, 0].set_title(f'{rotulos_transf[col]} — escala original', fontsize=10, pad=8)
    axes[i, 0].set_xlabel(f'Assimetria = {s.skew():.2f}', fontsize=9)

    axes[i, 1].hist(s_log, bins=50, density=True, color='darkorange',
                    edgecolor='white', alpha=0.82, label='Distribuição observada')
    mu, sigma = s_log.mean(), s_log.std(ddof=0)
    x = np.linspace(s_log.min(), s_log.max(), 300)
    axes[i, 1].plot(x, (1/(sigma*np.sqrt(2*np.pi)))*np.exp(-0.5*((x-mu)/sigma)**2),
                    color='firebrick', lw=1.8, label='Curva normal de referência')
    axes[i, 1].set_title(f'{rotulos_transf[col]} — escala log10', fontsize=10, pad=8)
    axes[i, 1].set_xlabel(f'Assimetria = {s_log.skew():.2f}', fontsize=9)

    for ax in axes[i]:
        ax.set_ylabel('Densidade')
        ax.grid(axis='y', alpha=0.2)

axes[0, 1].legend(loc='upper right', fontsize=8, frameon=False)
fig.suptitle('Efeito da transformação logarítmica', fontsize=14, y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.97], h_pad=2.0, w_pad=1.5)
plt.show()


### Célula 20

Resume as transformações indicadas e a assimetria obtida após sua aplicação.

In [ ]:
assimetria_transformada = {
    col: np.log10(df.loc[df[col] > 0, col].dropna()).skew()
    for col in ['price', 'lsqft', 'bsqft']
}
assimetria_transformada.update({
    'br': df['br'].dropna().skew(),
    'year': df['year'].dropna().skew()
})

tabela_transformacoes = pd.DataFrame({
    'Variável': ['price', 'lsqft', 'bsqft', 'br', 'year'],
    'Forma observada': [
        'Assimetria à direita',
        'Assimetria acentuada à direita',
        'Assimetria à direita',
        'Assimetria leve à direita',
        'Assimetria à esquerda'
    ],
    'Alternativa': [
        'log10', 'log10', 'log10',
        'manter inicialmente', 'manter inicialmente'
    ],
    'Assimetria após a transformação': [
        f"{assimetria_transformada[col]:.2f}".replace('.', ',')
        for col in ['price', 'lsqft', 'bsqft', 'br', 'year']
    ]
})

print('Transformações promissoras com base na Tabela 3 do resumo:')
display(
    tabela_transformacoes.style
    .hide(axis='index')
    .set_properties(**{'text-align': 'left'})
    .set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'left'),
                                     ('background-color', '#eeeeee'),
                                     ('color', '#222222')]},
        {'selector': 'td', 'props': [('border', '1px solid #d0d0d0')]}
    ])
)

### Célula 21

Agrupa vendas e preços por CEP.

In [ ]:
df['zip_str'] = df['zip'].dropna().astype(int).astype(str).str.zfill(5)

zip_stats = df.groupby('zip_str').agg(
    vendas        = ('price', 'count'),
    preco_mediano = ('price', 'median'),
    preco_medio   = ('price', 'mean')
).reset_index()

print(f'CEPs únicos: {len(zip_stats)}')
print(f'Registros sem CEP: {df["zip"].isna().sum()}')
zip_stats.sort_values('vendas', ascending=False).head(10)


### Célula 22

Baixa as coordenadas e associa os CEPs à localização.

In [ ]:
import ssl, urllib.request, zipfile, io, os

GEO_FILE = 'us_zip_coords.csv'

if not os.path.exists(GEO_FILE):
    print('Baixando base GeoNames US ZIP codes...')
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    with urllib.request.urlopen('https://download.geonames.org/export/zip/US.zip',
                                context=ctx) as resp:
        raw = resp.read()
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        with z.open('US.txt') as f:
            geo_base = pd.read_csv(
                f, sep='\t', header=None,
                names=['country','zip','place','state','state_code',
                       'county','county_code','community','community_code',
                       'lat','lon','accuracy']
            )
    geo_base.to_csv(GEO_FILE, index=False)
    print(f'Base salva: {len(geo_base):,} registros')
else:
    geo_base = pd.read_csv(GEO_FILE, dtype={'zip': str})
    print(f'Base GeoNames carregada do cache: {len(geo_base):,} registros')

geo_base['zip'] = geo_base['zip'].astype(str).str.zfill(5)
geo_lookup = geo_base[['zip','place','state_code','lat','lon']].drop_duplicates('zip')

zip_geo = zip_stats.merge(geo_lookup, left_on='zip_str', right_on='zip', how='left')
zip_geo_ok = zip_geo.dropna(subset=['lat','lon']).copy()

print(f'\nCEPs geocodificados: {len(zip_geo_ok)} / {len(zip_stats)}')
zip_geo_ok[['zip_str','place','state_code','lat','lon','vendas','preco_mediano']].head(5)


### Célula 23

Gera os mapas estático e interativo e salva o arquivo HTML.

In [ ]:
import folium
from folium.plugins import HeatMap
from IPython.display import display

fig, axes = plt.subplots(1, 2, figsize=(12, 5.8), sharex=True, sharey=True)

heat = axes[0].hexbin(
    zip_geo_ok['lon'], zip_geo_ok['lat'], C=zip_geo_ok['vendas'],
    reduce_C_function=np.sum, gridsize=32, mincnt=1, cmap='YlOrRd',
    linewidths=0.15, edgecolors='white'
)
cbar_heat = fig.colorbar(heat, ax=axes[0], pad=0.015, shrink=0.86)
cbar_heat.set_label('Quantidade de vendas')
axes[0].set_title('Mapa de calor — volume de vendas', fontsize=11, pad=10)

sizes = 18 + 150 * np.sqrt(zip_geo_ok['vendas'] / zip_geo_ok['vendas'].max())
points = axes[1].scatter(
    zip_geo_ok['lon'], zip_geo_ok['lat'], c=zip_geo_ok['preco_mediano'],
    s=sizes, cmap='coolwarm', alpha=0.76, edgecolor='white', linewidth=0.35
)
cbar_price = fig.colorbar(points, ax=axes[1], pad=0.015, shrink=0.86)
cbar_price.set_label('Preço mediano (US$)')
axes[1].set_title('Preço mediano por CEP', fontsize=11, pad=10)
axes[1].text(
    0.02, 0.02, 'Cor: preço mediano\nTamanho: volume de vendas',
    transform=axes[1].transAxes, fontsize=8, va='bottom',
    bbox={'boxstyle': 'round,pad=0.3', 'facecolor': 'white',
          'edgecolor': '#bbbbbb', 'alpha': 0.9}
)

for ax in axes:
    ax.set_xlabel('Longitude')
    ax.grid(alpha=0.18)
axes[0].set_ylabel('Latitude')
fig.suptitle('Distribuição geográfica das vendas por CEP — 2005', fontsize=14, y=0.99)
plt.tight_layout(rect=[0, 0, 1, 0.95], w_pad=1.8)
plt.show()

center = [zip_geo_ok['lat'].mean(), zip_geo_ok['lon'].mean()]
m = folium.Map(location=center, zoom_start=9, tiles='CartoDB positron')
heat_group = folium.FeatureGroup(name='Mapa de calor — volume de vendas', show=True)
HeatMap([[r['lat'], r['lon'], r['vendas']] for _, r in zip_geo_ok.iterrows()],
        min_opacity=0.3, radius=18, blur=12).add_to(heat_group)
heat_group.add_to(m)

price_min = zip_geo_ok['preco_mediano'].min()
price_max = zip_geo_ok['preco_mediano'].max()
fg = folium.FeatureGroup(name='Preço mediano por CEP', show=True)
for _, row in zip_geo_ok.iterrows():
    t = (row['preco_mediano'] - price_min) / (price_max - price_min)
    color = f'hsl({int((1-t)*240)}, 80%, 45%)'
    radius = 4 + (row['vendas'] / zip_geo_ok['vendas'].max()) * 14
    popup = (f"<b>CEP {row['zip_str']}</b><br>{row['place']}, {row['state_code']}<br>"
             f"Vendas em 2005: <b>{int(row['vendas'])}</b><br>"
             f"Preço mediano: <b>US$ {row['preco_mediano']:,.0f}</b>")
    folium.CircleMarker(location=[row['lat'], row['lon']], radius=radius,
                        color=color, fill=True, fill_color=color, fill_opacity=0.7,
                        popup=folium.Popup(popup, max_width=240),
                        tooltip=f"CEP {row['zip_str']} | US$ {row['preco_mediano']:,.0f}").add_to(fg)
fg.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.get_root().html.add_child(folium.Element("""
<div style="position:fixed;bottom:28px;left:28px;z-index:1000;
            background:white;padding:11px 13px;border-radius:7px;
            border:1px solid #999;box-shadow:0 1px 5px rgba(0,0,0,.25);
            font-size:12px;line-height:1.55">
  <b>Legenda — preço mediano por CEP</b><br>
  <span style="color:hsl(240,80%,45%);font-size:17px">&#9679;</span> Menor preço<br>
  <span style="color:hsl(120,80%,45%);font-size:17px">&#9679;</span> Preço intermediário<br>
  <span style="color:hsl(0,80%,45%);font-size:17px">&#9679;</span> Maior preço<br>
  <span style="color:#555">Tamanho do círculo = volume de vendas</span><br>
  <i>Use o controle no canto superior direito<br>para ativar ou ocultar as camadas.</i>
</div>
"""))
m.save('mapa_imoveis_2005.html')
print('Mapa interativo salvo: mapa_imoveis_2005.html')
display(m)
